# Automatic Mask Generation

In automatic mode, SAM densely samples a grid of points across the image, generates candidate masks for each, and then filters and deduplicates them. The result is a full-image segmentation without any manual prompts.

This is useful for unsupervised object inventory: counting and mapping all distinct objects in a scene.

## References
- samgeo docs: https://samgeo.gishub.org/samgeo/
- SAM paper: https://arxiv.org/abs/2304.02643

## 1. Setup

In [ ]:
import os
import urllib.request
import matplotlib.pyplot as plt
import rasterio
from rasterio.plot import show
from samgeo import SamGeo

url = "https://github.com/opengeos/samgeo/releases/download/v0.2.0/aerial_image.tif"
image_path = "aerial_image.tif"
if not os.path.exists(image_path):
    urllib.request.urlretrieve(url, image_path)

# automatic=True enables the dense grid sampling mode
sam = SamGeo(
    model_type="vit_b",
    automatic=True,
    points_per_side=16,       # grid density (higher = more masks, slower)
    pred_iou_thresh=0.86,     # discard low-quality masks
    stability_score_thresh=0.92,
    min_mask_region_area=100, # discard tiny mask fragments (pixels)
)

## 2. Generate Masks for the Entire Image

In [ ]:
sam.generate(image_path, output="auto_masks.tif", foreground=True, unique=True)
print("Mask generation complete. Output saved to auto_masks.tif")

## 3. Visualize Mask Overlay

In [ ]:
sam.show_anns(
    cmap="Paired",
    alpha=0.4,
    title="Automatic mask generation",
    blend=True,
    output="auto_masks_viz.png",
)

img = plt.imread("auto_masks_viz.png")
plt.figure(figsize=(10, 10))
plt.imshow(img)
plt.axis("off")
plt.tight_layout()
plt.show()

## 4. Inspect Mask Statistics

In [ ]:
import numpy as np

with rasterio.open("auto_masks.tif") as src:
    mask_arr = src.read(1)

unique_ids = np.unique(mask_arr)
unique_ids = unique_ids[unique_ids != 0]  # exclude background

areas = [np.sum(mask_arr == uid) for uid in unique_ids]

print(f"Number of detected objects: {len(unique_ids)}")
print(f"Median object area: {np.median(areas):.0f} px")
print(f"Largest object: {max(areas)} px")
print(f"Smallest object: {min(areas)} px")

plt.figure(figsize=(7, 4))
plt.hist(areas, bins=30, edgecolor="black")
plt.xlabel("Object area (pixels)")
plt.ylabel("Count")
plt.title("Distribution of detected object areas")
plt.tight_layout()
plt.show()

## 5. Export Masks as GeoJSON

Each mask becomes a polygon feature with its area attribute preserved.

In [ ]:
sam.tiff_to_vector("auto_masks.tif", "auto_masks.geojson")
print("Exported masks to auto_masks.geojson")

## 6. Tuning Tips

| Parameter | Effect |
|---|---|
| `points_per_side` | Higher → more masks found, slower runtime |
| `pred_iou_thresh` | Higher → only keep high-confidence masks |
| `stability_score_thresh` | Higher → discard unstable (noisy) masks |
| `min_mask_region_area` | Higher → discard small fragments |

Start with defaults and increase `pred_iou_thresh` / `stability_score_thresh` if you see too many false positives.